In [1]:
import pyro
import pyro.distributions as dist

from pyro.nn import PyroSample
from pyro.infer.autoguide import AutoDiagonalNormal
from pyro.infer import SVI, Trace_ELBO
from pgmpy.parameter._base import BaseParameter
from skpro.distributions.normal import Normal as SkproNormal
from torch import nn
from pyro.nn import PyroModule
from pyro.nn import PyroSample

from pyro.infer import Predictive
import os
from functools import partial
import torch
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

pyro.set_rng_seed(1)

%matplotlib inline
plt.style.use('default')


c:\Users\eogus\anaconda3\envs\pgmpy_311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:

import math
from copy import deepcopy

import pandas as pd
import torch
import pyro
import pyro.poutine as poutine

from pyro.infer import SVI, Trace_ELBO
from skpro.distributions import Mixture


class BayesianFunctionalRegression(BaseParameter):
    def __init__(
        self,
        model,
        guide,
        converter,
        num_iterations=1500,
        lr=0.03,
        posterior_samples=2000,
    ):
        super().__init__()

        self.num_iterations = num_iterations
        self.lr = lr
        self.posterior_samples = posterior_samples

        self.model = model if model else None
        self.guide = guide if guide else None
        self.converter = converter if converter else None

        self._is_fitted = False


    def fit(self, X, y):
        if not callable(self.model):
            raise TypeError("model must be callable.")

        if not callable(self.guide):
            raise TypeError("guide must be callable.")

        if not callable(self.converter):
            raise TypeError("converter must be callable.")

        self._is_fitted = False

        # model 또는 guide의 파라미터로 dtype/device를 결정한다.
        reference_param = None

        for obj in (self.guide, self.model):
            parameters = getattr(obj, "parameters", None)

            if callable(parameters):
                try:
                    reference_param = next(parameters())
                    break
                except StopIteration:
                    pass

        if reference_param is not None:
            self._torch_device = reference_param.device
            self._torch_dtype = (
                reference_param.dtype
                if reference_param.is_floating_point()
                else torch.get_default_dtype()
            )
        elif torch.is_tensor(X):
            self._torch_device = X.device
            self._torch_dtype = (
                X.dtype if X.is_floating_point() else torch.get_default_dtype()
            )
        else:
            self._torch_device = torch.device("cpu")
            self._torch_dtype = torch.get_default_dtype()

        def to_tensor(data):
            if torch.is_tensor(data):
                return data.detach().to(
                    device=self._torch_device,
                    dtype=self._torch_dtype,
                )

            if isinstance(data, (pd.DataFrame, pd.Series)):
                data = data.to_numpy(dtype=float)

            return torch.as_tensor(
                data,
                dtype=self._torch_dtype,
                device=self._torch_device,
            )

        X_tensor = to_tensor(X)
        y_tensor = to_tensor(y)

        # 단일 feature 입력을 (n_samples, 1)로 정규화한다.
        if X_tensor.ndim == 1:
            X_tensor = X_tensor.reshape(-1, 1)

        if X_tensor.ndim != 2:
            raise ValueError(
                "X must be a two-dimensional tabular input. "
                f"Received shape={tuple(X_tensor.shape)}."
            )

        if y_tensor.ndim == 0 or y_tensor.ndim > 2:
            raise ValueError(
                "y must be one- or two-dimensional. "
                f"Received shape={tuple(y_tensor.shape)}."
            )

        if X_tensor.shape[0] != y_tensor.shape[0]:
            raise ValueError(
                "X and y must have the same number of rows. "
                f"Received {X_tensor.shape[0]} and {y_tensor.shape[0]}."
            )

        if X_tensor.shape[0] == 0:
            raise ValueError("X and y must contain at least one observation.")

        self.n_features_in_ = X_tensor.shape[1]

        if isinstance(X, pd.DataFrame):
            self.feature_names_in_ = X.columns.copy()

        if isinstance(y, pd.DataFrame):
            self._y_columns = y.columns.copy()
        elif isinstance(y, pd.Series):
            column_name = y.name if y.name is not None else "y"
            self._y_columns = pd.Index([column_name])
        else:
            n_outputs = 1 if y_tensor.ndim == 1 else y_tensor.shape[1]
            self._y_columns = pd.RangeIndex(n_outputs)

        for obj in (self.model, self.guide):
            train = getattr(obj, "train", None)
            if callable(train):
                train(True)

        param_store = pyro.get_param_store()
        self.loss_history_ = []

        # Pyro ParamStore는 전역 객체이므로 estimator별 scope에서 학습한다.
        with param_store.scope() as fitted_param_state:
            optimizer = pyro.optim.Adam({"lr": self.lr})

            svi = SVI(
                model=self.model,
                guide=self.guide,
                optim=optimizer,
                loss=Trace_ELBO(),
            )

            for iteration in range(self.num_iterations):
                loss = float(svi.step(X_tensor, y_tensor))

                if not math.isfinite(loss):
                    raise FloatingPointError(
                        "Non-finite ELBO loss encountered at "
                        f"iteration {iteration + 1}: {loss}."
                    )

                normalized_loss = loss / X_tensor.shape[0]
                self.loss_history_.append(normalized_loss)

                if iteration % 100 == 0:
                    print(
                        f"[iteration {iteration + 1:04d}] "
                        f"loss: {normalized_loss:.4f}"
                    )

        # scope 종료 후에도 학습된 ParamStore를 estimator 내부에 보관한다.
        self._param_store_state = deepcopy(fitted_param_state)

        for obj in (self.model, self.guide):
            eval_method = getattr(obj, "eval", None)
            if callable(eval_method):
                eval_method()

        self._is_fitted = True
        return self


    def predict_proba(self, X, num_samples=None):
        if not self._is_fitted:
            raise RuntimeError(
                "This BayesianFunctionalRegression instance is not fitted yet."
            )

        if num_samples is None:
            num_samples = self.posterior_samples

        if not isinstance(num_samples, int) or num_samples <= 0:
            raise ValueError("num_samples must be a positive integer.")

        if isinstance(X, pd.DataFrame):
            index = X.index.copy()
            X_values = X.to_numpy(dtype=float)
        elif isinstance(X, pd.Series):
            index = X.index.copy()
            X_values = X.to_numpy(dtype=float)
        elif torch.is_tensor(X):
            index = pd.RangeIndex(X.shape[0])
            X_values = X
        else:
            index = pd.RangeIndex(len(X))
            X_values = X

        if torch.is_tensor(X_values):
            X_tensor = X_values.detach().to(
                device=self._torch_device,
                dtype=self._torch_dtype,
            )
        else:
            X_tensor = torch.as_tensor(
                X_values,
                dtype=self._torch_dtype,
                device=self._torch_device,
            )

        if X_tensor.ndim == 1:
            X_tensor = X_tensor.reshape(-1, 1)

        if X_tensor.ndim != 2:
            raise ValueError(
                "X must be a two-dimensional tabular input. "
                f"Received shape={tuple(X_tensor.shape)}."
            )

        if X_tensor.shape[1] != self.n_features_in_:
            raise ValueError(
                "The number of features differs from fit. "
                f"Expected {self.n_features_in_}, "
                f"received {X_tensor.shape[1]}."
            )

        for obj in (self.model, self.guide):
            eval_method = getattr(obj, "eval", None)
            if callable(eval_method):
                eval_method()

        components = []
        param_store = pyro.get_param_store()

        # 저장된 estimator 전용 ParamStore를 임시로 복원한다.
        with param_store.scope(deepcopy(self._param_store_state)):
            with torch.no_grad():
                for _ in range(num_samples):
                    # q(z | D)에서 latent parameter를 한 번 추출한다.
                    guide_trace = poutine.trace(self.guide).get_trace(
                        X_tensor,
                        None,
                    )

                    # 추출한 latent parameter를 모델에 주입한다.
                    replayed_model = poutine.replay(
                        self.model,
                        trace=guide_trace,
                    )

                    model_trace = poutine.trace(replayed_model).get_trace(
                        X_tensor,
                        None,
                    )

                    if "obs" not in model_trace.nodes:
                        raise KeyError(
                            "The Pyro model must define a sample site named 'obs'."
                        )

                    obs_site = model_trace.nodes["obs"]

                    if obs_site.get("type") != "sample":
                        raise TypeError(
                            "The 'obs' site must be a pyro.sample site."
                        )

                    # Tensor sample이 아니라 조건부 Pyro Distribution 객체이다.
                    pyro_distribution = obs_site["fn"]

                    skpro_distribution = self.converter(
                        pyro_distribution,
                        index=index,
                        columns=self._y_columns,
                    )

                    components.append(skpro_distribution)

        # p(y* | X*, D)
        # ≈ 1/S sum_s p(y* | X*, z_s), z_s ~ q(z | D)
        return Mixture(
            distributions=components,
            weights=None,
            indep_rows=False,
            indep_cols=False,
            index=index,
            columns=self._y_columns,
        )


In [6]:
DATA_URL = "https://github.com/pyro-ppl/datasets/blob/master/rugged_data.csv?raw=true"
data = pd.read_csv(DATA_URL, encoding="ISO-8859-1")
df = data[["cont_africa", "rugged", "rgdppc_2000"]]
df = df[np.isfinite(df.rgdppc_2000)]
df["rgdppc_2000"] = np.log(df["rgdppc_2000"])

# Dataset: Add a feature to capture the interaction between "cont_africa" and "rugged"
df["cont_africa_x_rugged"] = df["cont_africa"] * df["rugged"]
data = torch.tensor(df[["cont_africa", "rugged", "cont_africa_x_rugged", "rgdppc_2000"]].values,
                        dtype=torch.float)
x_data, y_data = data[:, :-1], data[:, -1]


In [ ]:
import torch
import pyro
import pyro.distributions as dist

def nonlinear_student_t_model(X, y=None):
    """Nonlinear heteroscedastic Student-t regression.

    Parameters
    ----------
    X : torch.Tensor, shape (n_samples, 2)
    y : torch.Tensor or None
        shape (n_samples,) 또는 (n_samples, 1)
    """
    if X.ndim != 2 or X.shape[1] != 2:
        raise ValueError(
            "This example requires X with exactly two features. "
            f"Received shape={tuple(X.shape)}."
        )

    if y is not None:
        if y.ndim == 2:
            if y.shape[1] != 1:
                raise ValueError(
                    "This example supports a single target column."
                )
            y = y.squeeze(-1)

        if y.ndim != 1:
            raise ValueError(
                "y must have shape (n_samples,) or (n_samples, 1)."
            )

    x1 = X[:, 0]
    x2 = X[:, 1]

    # 양수 parameter
    amplitude = pyro.sample(
        "amplitude",
        dist.LogNormal(
            X.new_tensor(0.0),
            X.new_tensor(0.5),
        ),
    )

    frequency = pyro.sample(
        "frequency",
        dist.LogNormal(
            X.new_tensor(0.0),
            X.new_tensor(0.35),
        ),
    )

    # 실수 전체를 support로 가지는 parameter
    phase = pyro.sample(
        "phase",
        dist.Laplace(
            X.new_tensor(0.0),
            X.new_tensor(1.0),
        ),
    )

    offset = pyro.sample(
        "offset",
        dist.Laplace(
            X.new_tensor(0.0),
            X.new_tensor(2.0),
        ),
    )

    quadratic = pyro.sample(
        "quadratic",
        dist.Laplace(
            X.new_tensor(0.0),
            X.new_tensor(0.5),
        ),
    )

    interaction = pyro.sample(
        "interaction",
        dist.Laplace(
            X.new_tensor(0.0),
            X.new_tensor(1.0),
        ),
    )

    # Student-t likelihood의 기본 scale
    base_scale = pyro.sample(
        "base_scale",
        dist.HalfCauchy(
            X.new_tensor(0.5),
        ),
    )

    # x2에 따른 heteroscedasticity 크기
    hetero_strength = pyro.sample(
        "hetero_strength",
        dist.LogNormal(
            X.new_tensor(-1.0),
            X.new_tensor(0.5),
        ),
    )

    # df = 2 + 양수 값으로 두어 분산이 존재하도록 구성한다.
    df_minus_two = pyro.sample(
        "df_minus_two",
        dist.Gamma(
            concentration=X.new_tensor(2.0),
            rate=X.new_tensor(0.5),
        ),
    )
    degrees_of_freedom = df_minus_two + 2.0

    # 비선형 평균함수
    mean = (
        offset
        + amplitude * torch.sin(frequency * x1 + phase)
        + quadratic * x1.square()
        + interaction * torch.tanh(x1 * x2)
    )

    # 입력에 따라 관측 불확실성이 변한다.
    scale = base_scale * (
        1.0
        + hetero_strength * torch.sigmoid(x2)
    )

    with pyro.plate("data", X.shape[0]):
        pyro.sample(
            "obs",
            dist.StudentT(
                df=degrees_of_freedom,
                loc=mean,
                scale=scale,
            ),
            obs=y,
        )


In [ ]:
import math

from torch.distributions import constraints


def nonlinear_student_t_guide(X, y=None):
    """Non-Normal mean-field variational guide."""

    def unconstrained_param(name, initial_value):
        return pyro.param(
            name,
            lambda: X.new_tensor(initial_value),
        )

    def positive_param(name, initial_value):
        return pyro.param(
            name,
            lambda: X.new_tensor(initial_value),
            constraint=constraints.positive,
        )

    def sample_lognormal(site_name, initial_value, initial_scale=0.2):
        log_loc = unconstrained_param(
            f"{site_name}_q_log_loc",
            math.log(initial_value),
        )
        log_scale = positive_param(
            f"{site_name}_q_log_scale",
            initial_scale,
        )

        return pyro.sample(
            site_name,
            dist.LogNormal(
                loc=log_loc,
                scale=log_scale,
            ),
        )

    def sample_laplace(site_name, initial_loc=0.0, initial_scale=0.2):
        loc = unconstrained_param(
            f"{site_name}_q_loc",
            initial_loc,
        )
        scale = positive_param(
            f"{site_name}_q_scale",
            initial_scale,
        )

        return pyro.sample(
            site_name,
            dist.Laplace(
                loc=loc,
                scale=scale,
            ),
        )

    # 양수 latent variables
    sample_lognormal(
        site_name="amplitude",
        initial_value=1.0,
    )

    sample_lognormal(
        site_name="frequency",
        initial_value=1.0,
    )

    sample_lognormal(
        site_name="base_scale",
        initial_value=0.5,
    )

    sample_lognormal(
        site_name="hetero_strength",
        initial_value=0.3,
    )

    sample_lognormal(
        site_name="df_minus_two",
        initial_value=4.0,
        initial_scale=0.25,
    )

    # 실수 latent variables
    sample_laplace(
        site_name="phase",
        initial_loc=0.0,
    )

    sample_laplace(
        site_name="offset",
        initial_loc=0.0,
    )

    sample_laplace(
        site_name="quadratic",
        initial_loc=0.0,
    )

    sample_laplace(
        site_name="interaction",
        initial_loc=0.0,
    )


In [32]:
import numpy as np

from skpro.distributions import TDistribution


def student_t_converter(
    pyro_distribution,
    *,
    index,
    columns,
):
    """Convert Pyro StudentT to skpro TDistribution.

    이 converter는 단일 출력 회귀를 대상으로 한다.
    """
    original_distribution = pyro_distribution
    distribution = pyro_distribution

    required_parameters = ("df", "loc", "scale")

    # plate, masking 또는 Independent 등에 의해
    # distribution이 wrapper로 감싸진 경우 base_dist를 탐색한다.
    while not all(
        hasattr(distribution, parameter)
        for parameter in required_parameters
    ):
        if not hasattr(distribution, "base_dist"):
            raise TypeError(
                "Expected a StudentT-like Pyro distribution exposing "
                "'df', 'loc', and 'scale'. "
                f"Received {type(pyro_distribution).__name__}."
            )

        distribution = distribution.base_dist

    df = distribution.df.detach()
    loc = distribution.loc.detach()
    scale = distribution.scale.detach()

    # 세 parameter의 shape을 맞춘다.
    df, loc, scale = torch.broadcast_tensors(
        df,
        loc,
        scale,
    )

    # ExpandedDistribution인 경우 원래 batch_shape까지 확장한다.
    target_batch_shape = tuple(original_distribution.batch_shape)

    if target_batch_shape:
        df = torch.broadcast_to(df, target_batch_shape)
        loc = torch.broadcast_to(loc, target_batch_shape)
        scale = torch.broadcast_to(scale, target_batch_shape)

    df = df.cpu()
    loc = loc.cpu()
    scale = scale.cpu()

    n_rows = len(index)
    n_columns = len(columns)

    expected_size = n_rows * n_columns

    def reshape_parameter(parameter, name):
        if parameter.numel() == 1:
            parameter = parameter.expand(expected_size)

        if parameter.numel() != expected_size:
            raise ValueError(
                f"Pyro StudentT parameter '{name}' has an incompatible "
                f"shape {tuple(parameter.shape)}. Expected {expected_size} "
                "elements for the requested index and columns."
            )

        return parameter.reshape(
            n_rows,
            n_columns,
        ).numpy()

    mu = reshape_parameter(loc, "loc")
    sigma = reshape_parameter(scale, "scale")
    degrees_of_freedom = reshape_parameter(df, "df")

    # 수치적으로 0인 scale을 방지한다.
    sigma = np.maximum(
        sigma,
        np.finfo(sigma.dtype).tiny,
    )

    degrees_of_freedom = np.maximum(
        degrees_of_freedom,
        np.finfo(degrees_of_freedom.dtype).tiny,
    )

    return TDistribution(
        mu=mu,
        sigma=sigma,
        df=degrees_of_freedom,
        index=index,
        columns=columns,
    )


In [33]:
import numpy as np
import pandas as pd


rng = np.random.default_rng(42)

n_samples = 500

x1 = rng.uniform(-3.0, 3.0, size=n_samples)
x2 = rng.normal(size=n_samples)

X = pd.DataFrame(
    {
        "x1": x1,
        "x2": x2,
    }
)

true_mean = (
    0.7
    + 1.8 * np.sin(1.4 * x1 + 0.3)
    + 0.25 * x1**2
    - 1.1 * np.tanh(x1 * x2)
)

true_scale = 0.25 * (
    1.0
    + 1.2 / (1.0 + np.exp(-x2))
)

noise = true_scale * rng.standard_t(
    df=4.0,
    size=n_samples,
)

y = pd.DataFrame(
    {
        "target": true_mean + noise,
    }
)


In [34]:
regressor = BayesianFunctionalRegression(
    model=nonlinear_student_t_model,
    guide=nonlinear_student_t_guide,
    converter=student_t_converter,
    num_iterations=3000,
    lr=0.01,
    posterior_samples=500,
)

regressor.fit(X, y)


[iteration 0001] loss: 3.8015
[iteration 0101] loss: 1.2072
[iteration 0201] loss: 1.7607
[iteration 0301] loss: 1.2551
[iteration 0401] loss: 0.9787
[iteration 0501] loss: 0.9577
[iteration 0601] loss: 1.3922
[iteration 0701] loss: 0.8629
[iteration 0801] loss: 0.9441
[iteration 0901] loss: 0.8905
[iteration 1001] loss: 0.8353
[iteration 1101] loss: 0.8399
[iteration 1201] loss: 0.8441
[iteration 1301] loss: 0.8606
[iteration 1401] loss: 0.8431
[iteration 1501] loss: 0.8306
[iteration 1601] loss: 0.8390
[iteration 1701] loss: 0.8579
[iteration 1801] loss: 0.8350
[iteration 1901] loss: 0.8347
[iteration 2001] loss: 0.8545
[iteration 2101] loss: 0.8396
[iteration 2201] loss: 0.8328
[iteration 2301] loss: 0.8394
[iteration 2401] loss: 0.8453
[iteration 2501] loss: 0.8419
[iteration 2601] loss: 0.8406
[iteration 2701] loss: 0.8466
[iteration 2801] loss: 0.8401
[iteration 2901] loss: 0.8357


BayesianFunctionalRegression(converter=<function student_t_converter at 0x0000020A7D8E7060>,
                             guide=<function nonlinear_student_t_guide at 0x0000020A784C7060>,
                             lr=0.01,
                             model=<function nonlinear_student_t_model at 0x0000020A784C5DA0>,
                             num_iterations=3000, posterior_samples=500)

In [35]:
pred_dist = regressor.predict_proba(X[:5])


In [36]:
pred_dist


Mixture(columns=Index(['target'], dtype='object'),
        distributions=[TDistribution(columns=Index(['target'], dtype='object'),
                                     df=array([[4.393296],
       [4.393296],
       [4.393296],
       [4.393296],
       [4.393296]], dtype=float32),
                                     index=RangeIndex(start=0, stop=5, step=1),
                                     mu=array([[1.182316 ],
       [0.3434166],
       [0.5577059],
       [2.786177 ],
       [1.6236194]], dtype=float32),
                                     sigma=array([[0.38885427],...
       [3.6253226],
       [3.6253226],
       [3.6253226],
       [3.6253226]], dtype=float32),
                                     index=RangeIndex(start=0, stop=5, step=1),
                                     mu=array([[1.1763756 ],
       [0.3606033 ],
       [0.53683233],
       [2.7691238 ],
       [1.514669  ]], dtype=float32),
                                     sigma=array([[0.4006256 ],
       [0.3653186 ],
       [0.3800081 ],
       [0.36072046],
       [0.35720798]], dtype=float32)), ...],
        indep_cols=False, indep_rows=False,
        index=RangeIndex(start=0, stop=5, step=1))